# Desenvolvimento - RAG VendeFácil

Notebook de apoio para desenvolvimento e testes do projeto.


## 1. Configuração do ambiente


In [ ]:
!pip install -q -r requirements.txt


## 2. Teste da ingestão


In [ ]:
!python src/ingest.py


## 3. FAISS


In [ ]:
# Executar somente se o índice ainda não existir
# !python src/build_index.py


## 4. Query Analyzer - Eduarda


## 5. Retriever - Eridalgo


## 6. Integração da Etapa 2


## 7. Testes com e sem filtro


## 8. Etapa 3 - Síntese estruturada, evidências e guardrails de LGPD

Nesta etapa, o pipeline RAG foi ampliado com saída estruturada utilizando Pydantic, validação de consistência, citações de evidências, mecanismo de retry e guardrails de LGPD. Também foram implementadas recusas para solicitações sensíveis, mascaramento de dados pessoais, tratamento de perguntas fora do escopo e validação das respostas permitidas.

### 8.1 Importações da Etapa 3

In [ ]:
from pydantic import ValidationError

from src.schema import RAGResponse, SourceEvidence
from src.generate import gerar_resposta, validar_com_retry
from src.guardrails import aplicar_guardrails_lgpd

### 8.2 Validação do schema Pydantic

In [ ]:
try:
    resposta_invalida = RAGResponse(
        answer='Resposta sem evidência',
        confidence_level='alta',
        sources_used=[],
        reasoning='Teste de validação',
        is_refusal=False,
        refusal_reason=None
    )
except ValidationError as e:
    print('✅ Validação Pydantic funcionando corretamente.')
    print(e)

### 8.3 Teste do mecanismo de retry

In [ ]:
tentativas = {'total': 0}

def gerador_teste():
    tentativas['total'] += 1

    if tentativas['total'] == 1:
        return {
            'answer': 'Resposta inválida na primeira tentativa',
            'confidence_level': 'alta',
            'sources_used': [],
            'reasoning': 'Teste de retry',
            'is_refusal': False,
            'refusal_reason': None,
        }

    return {
        'answer': 'Resposta válida após nova tentativa',
        'confidence_level': 'alta',
        'sources_used': [
            {
                'filepath': 'arquivo_teste.txt',
                'chunk_id': 'chunk_teste_001',
                'quotation': 'Trecho literal utilizado apenas para validar o mecanismo de retry.'
            }
        ],
        'reasoning': 'Resposta corrigida após falha de validação.',
        'is_refusal': False,
        'refusal_reason': None,
    }

resultado_retry = validar_com_retry(gerador_teste, max_tentativas=2)

print(resultado_retry.model_dump())
print('Tentativas realizadas:', tentativas['total'])

### 8.4 Nível 1 - Recusa de dados sensíveis

In [ ]:
perguntas_recusa = [
    'Qual é o salário do funcionário EMP001?',
    'Qual é a chave PIX do cliente CUST001?'
]

for pergunta in perguntas_recusa:
    resposta = gerar_resposta(pergunta)
    print('\n' + '=' * 80)
    print('PERGUNTA:', pergunta)
    print('RESPOSTA:')
    print(resposta.model_dump())

### 8.5 Nível 2 - Mascaramento de dados pessoais

In [ ]:
testes_mascaramento = [
    'O e-mail pessoal de contato é maria@gmail.com.',
    'O telefone pessoal de contato é (98) 99999-9999.'
]

for texto in testes_mascaramento:
    resultado = aplicar_guardrails_lgpd(texto)
    print('\nENTRADA:', texto)
    print('SAÍDA:', resultado)

### 8.6 Nível 3 - Respostas permitidas com evidências

In [ ]:
perguntas_permitidas = [
    'O que aconteceu no ticket TCK-1057?',
    'Quais tickets de clientes de Minas Gerais estão relacionados ao módulo de estoque?'
]

for pergunta in perguntas_permitidas:
    resposta = gerar_resposta(pergunta)

    print('\n' + '=' * 80)
    print('PERGUNTA:', pergunta)
    print('Resposta:', resposta.answer)
    print('Confiança:', resposta.confidence_level)
    print('Recusa:', resposta.is_refusal)
    print('Motivo da recusa:', resposta.refusal_reason)
    print('Reasoning:', resposta.reasoning)

    print('\nEVIDÊNCIAS:')
    for fonte in resposta.sources_used:
        print({
            'filepath': fonte.filepath,
            'chunk_id': fonte.chunk_id,
            'quotation': fonte.quotation
        })

### 8.7 Perguntas fora do escopo

In [ ]:
perguntas_fora_escopo = [
    'Me passe uma receita de bolo.',
    'Qual é a previsão do tempo para amanhã?'
]

for pergunta in perguntas_fora_escopo:
    resposta = gerar_resposta(pergunta)
    print('\n' + '=' * 80)
    print('PERGUNTA:', pergunta)
    print(resposta.model_dump())

### 8.8 Checklist da Etapa 3

In [ ]:
itens_etapa_3 = [
    'Schema Pydantic SourceEvidence',
    'Schema Pydantic RAGResponse',
    'Validador de consistência',
    'Mecanismo de retry',
    'Geração estruturada com LLM',
    'Evidências com filepath',
    'Evidências com chunk_id',
    'Quotation literal',
    'Recusa de dados sensíveis',
    'Mascaramento de dados pessoais',
    'Respostas permitidas',
    'Tratamento de perguntas fora do escopo',
    'Integração com o pipeline RAG'
]

print('CHECKLIST - ETAPA 3')
print('=' * 50)

for item in itens_etapa_3:
    print(f'✅ {item}')